# Cobblemon Atlas｜烈空坐骨骼姿勢疊加四組對照

同一個模型、相同固定鏡頭，分別輸出 **原始骨架(rest)**、**待機(ground_idle)**、**行走(ground_walk)**、**待機＋行走(idle_plus_walk)**。包含每組 24 幀動畫及 6 幀拼圖。所有結果只寫入 Drive 的 `reports/semantic-audit/pose-layer-experiment/`，不改動正式 WebP、網站或 checkpoint。

這是研究用對照：簡單相加不一定符合 Cobblemon 的最終 Poser 條件；請依結果再決定正確的疊加規則。

In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,json,zipfile,time
DRIVE_ROOT=Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK_ROOT=Path('/content/cobblemon-bedrock')
SITE_ROOT=Path('/content/cobblemon-atlas')
BRANCH='feat/animation-manifest-scanner-phase1'
FORMS=['0384_rayquaza']
ACTIONS=['ground_walk']
FPS,SECONDS,SIZE,SS=12,2,224,2
print('Isolated Rayquaza test:',FORMS,ACTIONS)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), f'找不到雲端資料夾 {DRIVE_ROOT}；請確認使用正確 Google 帳號'
for name in ['source_packs','manifests','renders','reports','checkpoints']:
    assert (DRIVE_ROOT/name).is_dir(), f'缺少 Drive 資料夾：{name}'
print('Google Drive 目錄確認成功:', DRIVE_ROOT)

In [ ]:
if not (SITE_ROOT/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE_ROOT)],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'sparse-checkout','set','tools','docs','notebooks','mega-showdown/data'],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'pull','--ff-only','origin',BRANCH],check=True)
sha=subprocess.check_output(['git','-C',str(SITE_ROOT),'rev-parse','--short','HEAD'],text=True).strip()
renderer_source=(SITE_ROOT/'tools/render_action_jobs.py').read_text()
assert 'def choose_actions(' in renderer_source, '目前載入的是舊版選擇器，請重新開啟 GitHub 最新 Notebook'
print('已載入最新版候選來源選擇器，Git commit:',sha)

In [ ]:
# 使用本機暫存磁碟渲染，避免每一幀反覆從 Drive 讀取大量小檔案。
WORK_ROOT.mkdir(parents=True,exist_ok=True)
packs = DRIVE_ROOT/'source_packs'
required = [packs/'development_resource_packs.zip', packs/'merged_pack.zip', packs/'render_portraits.py', DRIVE_ROOT/'manifests/animation-actions.json']
missing = [str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('請先上傳以下來源檔：\n'+'\n'.join(missing))
for archive in required[:2]:
    expected = WORK_ROOT/archive.stem
    if expected.is_dir():
        print('Reuse extracted pack:', expected)
        continue
    print('Extracting:', archive.name, flush=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            dest = (WORK_ROOT/item.filename).resolve()
            if not dest.is_relative_to(WORK_ROOT.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(WORK_ROOT)
    assert expected.is_dir(), f'壓縮檔最外層不是 {archive.stem}，請重新封裝'
(WORK_ROOT/'tools').mkdir(exist_ok=True)
shutil.copy2(packs/'render_portraits.py', WORK_ROOT/'tools/render_portraits.py')
manifest = DRIVE_ROOT/'manifests/animation-actions.json'
data=json.loads(manifest.read_text())
print('Manifest:',data['summary'])
print('Resources prepared locally')

In [ ]:
# 事前驗證：需要 GitHub 最新的四組對照工具和完整來源包
tool=SITE_ROOT/'tools/compare_pose_layers.py'
assert tool.is_file(), 'GitHub 程式版本太舊，請重新執行 git pull 那格'
assert (WORK_ROOT/'tools/render_portraits.py').is_file()
print('Ready for isolated pose experiment:',tool)

In [ ]:
# 不走 --sync-root：直接輸出到 Drive 的獨立實驗目錄，不修改正式 checkpoint。
output=DRIVE_ROOT/'reports/semantic-audit/pose-layer-experiment'
cmd=[sys.executable,str(tool),'--repo',str(WORK_ROOT),
     '--manifest',str(manifest),'--out',str(output),
     '--size','224','--fps','12','--seconds','2','--ss','2']
result=subprocess.run(cmd,check=False)
print('Exit code:',result.returncode,'output:',output)
if result.returncode:
    raise RuntimeError('對照實驗執行失敗，請提供上方錯誤訊息')
report=json.loads((output/'pose-layer-comparison.json').read_text())
for key,row in report['variants'].items():
    print(key,row['changed_pixels_max'],row['alpha_bbox'],row['min_margin_px'])
print('Contact sheets:',list(output.glob('*contact.png')))